In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

super_ai_engineer_season_6_ocr_2569_round2_path = kagglehub.competition_download('super-ai-engineer-season-6-ocr-2569-round2')

print('Data source import complete.')


In [ ]:
!pip install -q "google-genai>=1.0" pillow pandas tqdm opencv-python-headless

In [ ]:
import re
import json
import time
import pickle
from io import BytesIO
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm import tqdm

from google import genai
from google.genai import types as genai_types

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GEMINI_KEY = user_secrets.get_secret("GEMINI_API_KEY")

client = genai.Client(
    api_key=GEMINI_KEY,
    http_options={"api_version": "v1alpha"}
)

# MODEL = "gemini-3.1-pro-preview"
MODEL = "gemini-2.5-pro"

IMAGES_DIR = "/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/images"
TEMPLATE   = "/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/submission_template_v4.csv"
OUT_CSV    = "/kaggle/working/submission.csv"
CACHE_FILE = "/kaggle/working/cache_g31_crop.pkl"

RETRY_MAX = 3
RPM = 10
SAVE_DEBUG = False
DEBUG_DIR = Path("/kaggle/working/debug_crops")
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Ready | Model: {MODEL}")

✅ Ready | Model: gemini-2.5-pro


In [ ]:
_MULT = [
    ("ล้าน", 1_000_000),
    ("แสน", 100_000),
    ("หมื่น", 10_000),
    ("พัน", 1_000),
    ("ร้อย", 100),
    ("สิบ", 10),
]
_ONES = {
    "ศูนย์": 0, "หนึ่ง": 1, "เอ็ด": 1,
    "สอง": 2, "ยี่": 2, "สาม": 3, "สี่": 4,
    "ห้า": 5, "หก": 6, "เจ็ด": 7, "แปด": 8, "เก้า": 9,
}
_TOKS = sorted(list(_ONES.keys()) + [m[0] for m in _MULT], key=len, reverse=True)
THAI_DIGIT_MAP = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")

def thai_digits_to_int(text: str) -> int:
    if text is None:
        return 0
    digits = re.sub(r"[^\d]", "", str(text).translate(THAI_DIGIT_MAP))
    return int(digits) if digits else 0

def thai_words_to_int(text: str):
    if not text:
        return None
    t = re.sub(r"[\s,\(\)\.\-]", "", str(text))
    if not t:
        return None

    mult_map = dict(_MULT)

    def _chunk(s):
        total = 0
        pending = 0
        i = 0
        while i < len(s):
            tok = next((k for k in _TOKS if s.startswith(k, i)), None)
            if tok is None:
                return None
            if tok in _ONES:
                pending = _ONES[tok]
            else:
                mult = mult_map[tok]
                if tok == "สิบ" and pending == 0:
                    pending = 1
                total += pending * mult
                pending = 0
            i += len(tok)
        return total + pending

    if "ล้าน" not in t:
        return _chunk(t)

    parts = t.split("ล้าน")
    result = 0
    for i, p in enumerate(parts):
        v = _chunk(p)
        if v is None:
            return None
        if i < len(parts) - 1:
            result = (result + v) * 1_000_000
        else:
            result = result + v
    return result

def verify_and_choose(d_str: str, w_str: str):
    d_val = thai_digits_to_int(d_str) if d_str else 0
    w_val = thai_words_to_int(w_str) if w_str else None

    if w_val is not None and w_val > 0 and d_val == w_val:
        return d_val, "agree"
    if w_val is not None and w_val > 0:
        return w_val, "word_wins"
    if d_val > 0:
        return d_val, "digit_only"
    return 0, "zero"

assert thai_words_to_int("สามหมื่นสี่พันหนึ่งร้อยเจ็ดสิบเจ็ด") == 34177
assert thai_words_to_int("หนึ่งหมื่นสี่พันแปดร้อยสิบสาม") == 14813
assert thai_words_to_int("แปดสิบ") == 80
print("✅ Thai word parser OK")

✅ Thai word parser OK


In [ ]:
def get_pages(doc_id: str):
    base = Path(IMAGES_DIR)
    pages = []
    p1 = base / f"{doc_id}.png"
    if p1.exists():
        pages.append(p1)
    n = 2
    while True:
        pn = base / f"{doc_id}_page{n}.png"
        if not pn.exists():
            break
        pages.append(pn)
        n += 1
    return pages

def load_cv(path: Path) -> np.ndarray:
    arr = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"อ่านภาพไม่ได้: {path}")
    return img

def bgr_to_pil(img_bgr: np.ndarray) -> Image.Image:
    return Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))

In [ ]:
def find_table_bbox(img_bgr: np.ndarray):
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    bw = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        31, 15
    )

    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(40, h // 30)))
    hk = cv2.getStructuringElement(cv2.MORPH_RECT, (max(40, w // 20), 1))

    v = cv2.morphologyEx(bw, cv2.MORPH_OPEN, vk, iterations=1)
    hline = cv2.morphologyEx(bw, cv2.MORPH_OPEN, hk, iterations=1)
    grid = cv2.addWeighted(v, 0.5, hline, 0.5, 0.0)

    cnts, _ = cv2.findContours(grid, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_area = -1
    for c in cnts:
        x, y, ww, hh = cv2.boundingRect(c)
        area = ww * hh
        if ww > 0.55 * w and hh > 0.20 * h and area > best_area:
            best = (x, y, ww, hh)
            best_area = area

    if best is None:
        return (int(w * 0.08), int(h * 0.12), int(w * 0.90), int(h * 0.72))

    x, y, ww, hh = best
    pad_x = int(ww * 0.01)
    pad_y = int(hh * 0.01)

    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(w, x + ww + pad_x)
    y2 = min(h, y + hh + pad_y)
    return (x1, y1, x2 - x1, y2 - y1)

def cluster_positions(pos, gap=8):
    if len(pos) == 0:
        return []
    groups = []
    curr = [int(pos[0])]
    for p in pos[1:]:
        p = int(p)
        if p - curr[-1] <= gap:
            curr.append(p)
        else:
            groups.append((curr[0], curr[-1], int(np.mean(curr))))
            curr = [p]
    groups.append((curr[0], curr[-1], int(np.mean(curr))))
    return groups

def find_vote_column_bbox(table_bgr: np.ndarray):
    h, w = table_bgr.shape[:2]
    gray = cv2.cvtColor(table_bgr, cv2.COLOR_BGR2GRAY)

    bw = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        31, 15
    )

    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(30, h // 20)))
    v = cv2.morphologyEx(bw, cv2.MORPH_OPEN, vk, iterations=1)

    col_sum = v.sum(axis=0)
    if col_sum.max() <= 0:
        return (int(w * 0.68), 0, int(w * 0.30), h)

    line_cols = np.where(col_sum > col_sum.max() * 0.25)[0]
    groups = cluster_positions(line_cols, gap=10)

    if len(groups) >= 2:
        left_group = groups[-2]
        right_group = groups[-1]

        x1 = max(0, left_group[2] - 8)
        x2 = min(w, right_group[1] + 8)

        if (x2 - x1) < int(w * 0.12):
            x1 = int(w * 0.68)
            x2 = int(w * 0.98)
    else:
        x1 = int(w * 0.68)
        x2 = int(w * 0.98)

    return (x1, 0, x2 - x1, h)

def crop_vote_column(path: Path, save_debug=False):
    img = load_cv(path)
    tx, ty, tw, th = find_table_bbox(img)
    table = img[ty:ty+th, tx:tx+tw]

    vx, vy, vw, vh = find_vote_column_bbox(table)
    vote_col = table[vy:vy+vh, vx:vx+vw]

    if save_debug:
        stem = path.stem
        bgr_to_pil(table).save(DEBUG_DIR / f"{stem}_table.png")
        bgr_to_pil(vote_col).save(DEBUG_DIR / f"{stem}_vote_col_raw.png")

    return bgr_to_pil(vote_col), bgr_to_pil(table)

In [ ]:
def enhance_vote_crop(pil_img: Image.Image) -> Image.Image:
    img = np.array(pil_img.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    h, w = gray.shape
    scale = 2 if max(h, w) < 1800 else 1
    if scale > 1:
        gray = cv2.resize(gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)

    clahe = cv2.createCLAHE(clipLimit=2.8, tileGridSize=(8, 8))
    x = clahe.apply(gray)
    x = cv2.fastNlMeansDenoising(x, h=10)

    blur = cv2.GaussianBlur(x, (0, 0), 1.2)
    x = cv2.addWeighted(x, 1.6, blur, -0.6, 0)

    th = cv2.adaptiveThreshold(
        x, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31, 11
    )

    out = Image.fromarray(th).convert("RGB")
    out = ImageOps.expand(out, border=18, fill="white")
    return out

def normalize_for_llm(pil_img: Image.Image) -> Image.Image:
    img = pil_img.convert("RGB")
    arr = np.array(img)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=1.8, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    rgb = Image.fromarray(gray).convert("RGB")
    rgb = ImageOps.expand(rgb, border=18, fill="white")
    return rgb

def pil_to_part(pil_img: Image.Image, high=True):
    bio = BytesIO()
    pil_img.save(bio, format="PNG")
    return genai_types.Part.from_bytes(
        data=bio.getvalue(),
        mime_type="image/png",
        media_resolution=(
            genai_types.MediaResolution.MEDIA_RESOLUTION_HIGH
            if high else genai_types.MediaResolution.MEDIA_RESOLUTION_MEDIUM
        )
    )

In [ ]:
def make_prompt(expected_n: int, doc_type: str):
    if doc_type == "party_list":
        order_text = f"เรียงตามหมายเลขพรรคจากบนลงล่าง ต้องมีผู้สมัคร {expected_n} แถว"
    else:
        order_text = f"เรียงจากบนลงล่างตามลำดับแถวในตาราง ต้องมีผู้สมัคร {expected_n} แถว"

    return f"""
คุณคือผู้เชี่ยวชาญ OCR เอกสารเลือกตั้งภาษาไทย

ภาพที่ให้มาอาจมีหลายภาพของ "พื้นที่เดียวกัน":
- ภาพ raw
- ภาพ enhanced
- บางครั้งมีภาพ table ทั้งก้อนเป็นบริบท

งานของคุณ:
1) อ่านเฉพาะคอลัมน์ "ได้คะแนน"
2) สำหรับแต่ละผู้สมัคร ให้ดึง:
   - d = ตัวเลขไทยที่เห็น
   - w = คำอ่านในวงเล็บ
3) ต้องไม่รวม header
4) ต้องไม่รวมแถว "รวมคะแนนทั้งสิ้น" ใน rows
5) ต้องดึง bottom_total จากแถว "รวมคะแนนทั้งสิ้น" แยกต่างหาก

{order_text}

กฎการอ่าน:
- ถ้า d กับ w ไม่ตรงกัน ให้เชื่อ w มากกว่า
- ถ้าไม่มี w ให้ใส่ w=""
- ถ้าอ่านไม่ได้จริง ๆ ให้ d="" และ w=""
- อย่าข้ามแถว อย่าเพิ่มแถว
- rows ต้องยาว {expected_n} พอดี

ส่ง JSON เท่านั้น รูปแบบนี้:
{{
  "rows": [
    {{"d": "๓๔,๑๗๗", "w": "สามหมื่นสี่พันหนึ่งร้อยเจ็ดสิบเจ็ด"}},
    {{"d": "๑๔,๘๑๓", "w": "หนึ่งหมื่นสี่พันแปดร้อยสิบสาม"}}
  ],
  "bottom_total": {{"d": "๗๗,๐๗๕", "w": "เจ็ดหมื่นเจ็ดพันเจ็ดสิบห้า"}},
  "notes": []
}}
""".strip()

class RateLimiter:
    def __init__(self, rpm=10):
        self.interval = 60.0 / rpm
        self.last_time = 0.0

    def wait(self):
        delta = time.time() - self.last_time
        if delta < self.interval:
            time.sleep(self.interval - delta)
        self.last_time = time.time()

limiter = RateLimiter(RPM)

def call_gemini_json(parts, prompt):
    last_err = None
    for attempt in range(1, RETRY_MAX + 1):
        try:
            resp = client.models.generate_content(
                model=MODEL,
                contents=[prompt] + parts,
                config=genai_types.GenerateContentConfig(
                    response_mime_type="application/json",
                    thinking_config=genai_types.ThinkingConfig(
                        # thinking_level="high"
                        thinking_budget=8192
                    )
                    # ไม่กำหนด temperature ตามคำแนะนำ Gemini
                )
            )
            raw = re.sub(r"```(?:json)?\s*|\s*```", "", resp.text).strip()
            return json.loads(raw)
        except Exception as e:
            last_err = e
            wait_s = 60 if "429" in str(e) else 2 ** attempt
            print(f"  ↩️ attempt {attempt}: {e} | wait {wait_s}s")
            time.sleep(wait_s)
    raise RuntimeError(f"Gemini failed after retries: {last_err}")

In [ ]:
def parse_payload(data, expected_n: int):
    if isinstance(data, list):
        data = {"rows": data, "bottom_total": {}}

    rows = data.get("rows", [])
    bottom_total = data.get("bottom_total", {}) or {}

    votes = []
    sources = []

    for r in rows:
        d = str(r.get("d", "") or "")
        w = str(r.get("w", "") or "")
        val, src = verify_and_choose(d, w)
        votes.append(val)
        sources.append(src)

    td = str(bottom_total.get("d", "") or "")
    tw = str(bottom_total.get("w", "") or "")
    total_val, total_src = verify_and_choose(td, tw)

    meta = {
        "n_rows": len(votes),
        "sum_votes": int(sum(votes)),
        "bottom_total": int(total_val),
        "bottom_total_src": total_src,
        "ok_len": len(votes) == expected_n,
        "ok_total": (total_val > 0 and sum(votes) == total_val),
        "zero_rows": [i + 1 for i, v in enumerate(votes) if v == 0],
    }
    return votes, sources, meta

In [ ]:
def build_doc_parts(doc_id: str, doc_type: str, stage: int):
    """
    stage 1:
      - score column raw + enhanced
    stage 2:
      - score column raw + enhanced + full table crop
    """
    pages = get_pages(doc_id)
    if not pages:
        return []

    table_pages = pages

    parts = []

    for p in table_pages:
        vote_col_raw, table_raw = crop_vote_column(p, save_debug=SAVE_DEBUG)

        raw_norm = normalize_for_llm(vote_col_raw)
        enh_norm = enhance_vote_crop(vote_col_raw)

        if SAVE_DEBUG:
            stem = p.stem
            raw_norm.save(DEBUG_DIR / f"{stem}_vote_col_norm.png")
            enh_norm.save(DEBUG_DIR / f"{stem}_vote_col_enh.png")

        parts.append(pil_to_part(raw_norm, high=True))
        parts.append(pil_to_part(enh_norm, high=True))

        if stage >= 2:
            table_norm = normalize_for_llm(table_raw)
            if max(table_norm.size) > 1800:
                s = 1800 / max(table_norm.size)
                table_norm = table_norm.resize(
                    (int(table_norm.size[0] * s), int(table_norm.size[1] * s)),
                    Image.LANCZOS
                )
            parts.append(pil_to_part(table_norm, high=True))

    return parts

def extract_votes(doc_id: str, expected_n: int, doc_type: str):
    prompt = make_prompt(expected_n, doc_type)

    # ---------- Stage 1: crop only ----------
    parts = build_doc_parts(doc_id, doc_type, stage=1)
    if not parts:
        return [0] * expected_n, {"stage": 0, "ok": False, "reason": "no_images"}

    data = call_gemini_json(parts, prompt)
    votes, sources, meta = parse_payload(data, expected_n)

    if meta["ok_len"] and meta["ok_total"]:
        meta.update({"stage": 1, "ok": True, "reason": "crop_pass"})
        return votes, meta

    # ---------- Stage 2: crop + full table context ----------
    parts = build_doc_parts(doc_id, doc_type, stage=2)
    data2 = call_gemini_json(parts, prompt)
    votes2, sources2, meta2 = parse_payload(data2, expected_n)

    if meta2["ok_len"] and meta2["ok_total"]:
        meta2.update({"stage": 2, "ok": True, "reason": "context_pass"})
        return votes2, meta2

    cands = [
        (votes, meta),
        (votes2, meta2),
    ]
    cands = sorted(
        cands,
        key=lambda x: (
            x[1]["ok_len"],
            x[1]["ok_total"],
            -len(x[1]["zero_rows"]),
            -abs(x[1]["sum_votes"] - x[1]["bottom_total"]) if x[1]["bottom_total"] > 0 else -10**9
        ),
        reverse=True
    )

    final_votes, final_meta = cands[0]
    final_votes = (final_votes + [0] * expected_n)[:expected_n]
    final_meta.update({"stage": 99, "ok": False, "reason": "best_effort"})
    return final_votes, final_meta

In [ ]:
df = pd.read_csv(TEMPLATE)
df["doc_id"] = df["id"].apply(lambda x: x.rsplit("_", 1)[0])
df["row_num"] = df["id"].apply(lambda x: int(x.rsplit("_", 1)[1]))
df["doc_type"] = df["doc_id"].apply(
    lambda x: "party_list" if x.startswith("party_list") else "constituency"
)

grouped = df.groupby("doc_id")
doc_ids = sorted(df["doc_id"].unique())
print(f"📄 rows={len(df):,} | docs={len(doc_ids)}")

if Path(CACHE_FILE).exists():
    with open(CACHE_FILE, "rb") as f:
        cache = pickle.load(f)
    print(f"Cache loaded: {len(cache)}/{len(doc_ids)}")
else:
    cache = {}
    print("💾 Fresh start")

📄 rows=10,053 | docs=300
💾 Fresh start


In [ ]:
docs_with_img = [d for d in doc_ids if d not in cache and len(get_pages(d)) > 0]
docs_no_img = [d for d in doc_ids if d not in cache and len(get_pages(d)) == 0]

for doc_id in docs_no_img:
    n = len(grouped.get_group(doc_id))
    cache[doc_id] = {
        "votes": [0] * n,
        "meta": {"stage": 0, "ok": False, "reason": "no_images"}
    }

print(f"มีรูป: {len(docs_with_img)} | ไม่มีรูป: {len(docs_no_img)}")

for i, doc_id in enumerate(tqdm(docs_with_img, desc="OCR"), 1):
    grp = grouped.get_group(doc_id).sort_values("row_num")
    expected_n = len(grp)
    doc_type = grp["doc_type"].iloc[0]

    try:
        limiter.wait()
        votes, meta = extract_votes(doc_id, expected_n, doc_type)
        cache[doc_id] = {"votes": votes, "meta": meta}

        if i % 10 == 0:
            with open(CACHE_FILE, "wb") as f:
                pickle.dump(cache, f)

        flag = "✅" if meta.get("ok") else "⚠️"
        print(
            f"\n{flag} {doc_id} | stage={meta.get('stage')} "
            f"| rows={meta.get('n_rows')} "
            f"| sum={meta.get('sum_votes')} "
            f"| total={meta.get('bottom_total')} "
            f"| zeros={len(meta.get('zero_rows', []))}"
        )

    except Exception as e:
        print(f"\n{doc_id}: {e}")
        cache[doc_id] = {
            "votes": [0] * expected_n,
            "meta": {"stage": -1, "ok": False, "reason": str(e)}
        }

with open(CACHE_FILE, "wb") as f:
    pickle.dump(cache, f)

print("✅ OCR finished")

In [ ]:
vote_map = {}

for doc_id in doc_ids:
    grp = grouped.get_group(doc_id).sort_values("row_num")
    item = cache.get(doc_id, {"votes": [0] * len(grp)})
    votes = item["votes"]

    for i, (_, row) in enumerate(grp.iterrows()):
        vote_map[row["id"]] = int(votes[i]) if i < len(votes) else 0

df["votes"] = df["id"].map(vote_map).fillna(0).astype(int)
df[["id", "votes"]].to_csv(OUT_CSV, index=False)

nz = int((df["votes"] != 0).sum())
print(f"📊 total={len(df):,} | non-zero={nz:,} ({nz/len(df):.2%})")
print(f"✅ saved: {OUT_CSV}")
print(df[["id", "votes"]].head(20).to_string())

In [ ]:
def show_doc_meta(doc_id: str):
    item = cache.get(doc_id)
    if item is None:
        print("not found in cache")
        return
    print(json.dumps(item["meta"], ensure_ascii=False, indent=2))
    grp = grouped.get_group(doc_id).sort_values("row_num")
    for i, (_, row) in enumerate(grp.iterrows()):
        print(row["id"], item["votes"][i])

def rerun_one(doc_id: str):
    if doc_id in cache:
        del cache[doc_id]
    grp = grouped.get_group(doc_id).sort_values("row_num")
    limiter.wait()
    votes, meta = extract_votes(doc_id, len(grp), grp["doc_type"].iloc[0])
    cache[doc_id] = {"votes": votes, "meta": meta}
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(cache, f)
    show_doc_meta(doc_id)

In [ ]:
sub = pd.read_csv(OUT_CSV)
assert list(sub.columns) == ["id", "votes"]
assert len(sub) == 10053
assert pd.api.types.is_integer_dtype(sub["votes"])
print(f"{len(sub):,} rows OK")
print(sub.head(10).to_string())